# CUSTOMER CHURN PREDICTION 

In [1]:
!pip install pandas numpy scikit-learn matplotlib seaborn

# Loading Dataset

In [2]:
import pandas as pd

df = pd.read_csv("C:/Users/hp/Desktop/INTERN TASK/churn-bigml-20.csv")
df.head()

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn
0,LA,117,408,No,No,0,184.5,97,31.37,351.6,80,29.89,215.8,90,9.71,8.7,4,2.35,1,False
1,IN,65,415,No,No,0,129.1,137,21.95,228.5,83,19.42,208.8,111,9.40,12.7,6,3.43,4,True
2,NY,161,415,No,No,0,332.9,67,56.59,317.8,97,27.01,160.6,128,7.23,5.4,9,1.46,4,True
3,SC,111,415,No,No,0,110.4,103,18.77,137.3,102,11.67,189.6,105,8.53,7.7,6,2.08,2,False
4,HI,49,510,No,No,0,119.3,117,20.28,215.1,109,18.28,178.7,90,8.04,11.1,1,3.00,1,False


# Data Processing and Cleaning

In [3]:
df.info()
df.isnull().sum()
df = df.dropna()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 667 entries, 0 to 666
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   667 non-null    object 
 1   Account length          667 non-null    int64  
 2   Area code               667 non-null    int64  
 3   International plan      667 non-null    object 
 4   Voice mail plan         667 non-null    object 
 5   Number vmail messages   667 non-null    int64  
 6   Total day minutes       667 non-null    float64
 7   Total day calls         667 non-null    int64  
 8   Total day charge        667 non-null    float64
 9   Total eve minutes       667 non-null    float64
 10  Total eve calls         667 non-null    int64  
 11  Total eve charge        667 non-null    float64
 12  Total night minutes     667 non-null    float64
 13  Total night calls       667 non-null    int64  
 14  Total night charge      667 non-null    fl

In [14]:
# Clean column names
df.columns = df.columns

# Convert values to lowercase
df['international plan'] = df['international plan']
df['voice mail plan'] = df['voice mail plan']

# Fill missing values
df['churn'] = df['churn'].fillna(0).astype(int)

# Map plans
df['international plan'] = df['international plan'].map({'yes':1, 'no':0}).fillna(0).astype(int)
df['voice mail plan'] = df['voice mail plan'].map({'yes':1, 'no':0}).fillna(0).astype(int)

# Drop non-numeric
df = df.drop('state', axis=1, errors='ignore')

# Split
y = df['churn']
X = df.drop('churn', axis=1)

print("✅ Done successfully")

✅ Done successfully


In [21]:
X = X.apply(pd.to_numeric, errors='coerce').fillna(0)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("✅ Done successfully")

✅ Done successfully


In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)
print("✅ Done successfully")

✅ Done successfully


✅ Done successfully


In [28]:
# Train models again

# Logistic Regression
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Decision Tree
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

# Random Forest
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print("✅ Done successfully")

✅ Done successfully


In [29]:
evaluate_model(y_test, y_pred_lr, "Logistic Regression")
evaluate_model(y_test, y_pred_dt, "Decision Tree")
evaluate_model(y_test, y_pred_rf, "Random Forest")


🔍 Logistic Regression Performance:
Accuracy: 0.9029850746268657
Precision: 0.625
Recall: 0.3333333333333333
F1-score: 0.43478260869565216

🔍 Decision Tree Performance:
Accuracy: 0.8805970149253731
Precision: 0.4782608695652174
Recall: 0.7333333333333333
F1-score: 0.5789473684210527

🔍 Random Forest Performance:
Accuracy: 0.9552238805970149
Precision: 0.8461538461538461
Recall: 0.7333333333333333
F1-score: 0.7857142857142856


In [31]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)


Best Parameters: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}


In [32]:
best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test)

evaluate_model(y_test, y_pred_best, "Tuned Random Forest")


🔍 Tuned Random Forest Performance:
Accuracy: 0.9552238805970149
Precision: 0.8461538461538461
Recall: 0.7333333333333333
F1-score: 0.7857142857142856


In [33]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_best)
print("\nConfusion Matrix:\n", cm)


Confusion Matrix:
 [[117   2]
 [  4  11]]


#  Customer Churn Prediction Summary

Objective

To predict whether a customer will churn using classification models.

  Data Processing
Removed missing values
Converted categorical features (International Plan, Voice Mail Plan) to numerical values
Converted target variable (Churn) to 0/1
Dropped non-numeric column (State)
Scaled features using StandardScaler

Models Used
Logistic Regression
Decision Tree
Random Forest

Results
Logistic Regression
Accuracy: 0.903
F1-score: 0.435
Low recall (misses many churn cases)

Decision Tree
Accuracy: 0.881
F1-score: 0.579
Better recall but lower precision
Random Forest 
Accuracy: 0.955
F1-score: 0.786
Best balance of precision and recall

Hyperparameter Tuning
Best Parameters:
n_estimators = 100
max_depth = None
min_samples_split = 5
Tuned model performance remained strong and stable

Confusion Matrix Insight
Very few misclassifications
Model correctly identifies most churn and non-churn customers

Conclusion

Random Forest is the best performing model for churn prediction, offering high accuracy and balanced performance. It is suitable for customer retention analysis.
